# Notebook 02b — LayoutLMv3 Fine-tuning on FUNSD

Fine-tunes LayoutLMv3 on the FUNSD (Form Understanding in Noisy Scanned Documents) dataset.

This notebook:
1. Loads the FUNSD dataset from HuggingFace Hub
2. Tokenises with LayoutLMv3Processor (applies_ocr=True for raw image input)
3. Configures Trainer with TokenClassification head (4 labels: other/header/question/answer)
4. Trains with F1 metric on validation set, early stopping (patience=3)
5. Saves best checkpoint to models/layoutlmv3_finetuned/
6. Evaluates per-class F1 on test set

**Target:** val entity-level F1 >= 0.80 on FUNSD

**Requires:** CUDA GPU (8GB+ VRAM), `pip install seqeval`

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import torch
import numpy as np
from datasets import load_dataset
from transformers import (
    LayoutLMv3Processor,
    LayoutLMv3ForTokenClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
from seqeval.metrics import f1_score, precision_score, recall_score

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cpu':
    print('WARNING: Training on CPU will be very slow. Use a CUDA GPU.')

In [ ]:
# ── Load FUNSD from HuggingFace ───────────────────────────────────────────────
# FUNSD = Form Understanding in Noisy Scanned Documents
# Labels: O, B-HEADER, I-HEADER, B-QUESTION, I-QUESTION, B-ANSWER, I-ANSWER

dataset = load_dataset('nielsr/funsd-layoutlmv3', trust_remote_code=True)
print(dataset)

# Label mapping — must match src/layout.py FUNSD_LABELS
LABEL_LIST = dataset['train'].features['ner_tags'].feature.names
LABEL2ID = {l: i for i, l in enumerate(LABEL_LIST)}
ID2LABEL = {i: l for i, l in enumerate(LABEL_LIST)}
print(f'Labels: {LABEL_LIST}')

In [ ]:
# ── Processor and Tokenisation ───────────────────────────────────────────────
MODEL_NAME = 'microsoft/layoutlmv3-base'
processor = LayoutLMv3Processor.from_pretrained(MODEL_NAME, apply_ocr=False)

def preprocess(examples):
    images = examples['image']
    words  = examples['tokens']
    boxes  = examples['bboxes']
    ner_tags = examples['ner_tags']

    encoding = processor(
        images,
        words,
        boxes=boxes,
        word_labels=ner_tags,
        truncation=True,
        padding='max_length',
        max_length=512,
        return_tensors='pt',
    )
    return encoding

train_ds = dataset['train'].map(preprocess, batched=True, remove_columns=dataset['train'].column_names)
val_ds   = dataset['test'].map(preprocess,  batched=True, remove_columns=dataset['test'].column_names)
train_ds.set_format('torch')
val_ds.set_format('torch')

print(f'Train size: {len(train_ds)} | Val size: {len(val_ds)}')

In [ ]:
# ── Model ────────────────────────────────────────────────────────────────────
model = LayoutLMv3ForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABEL_LIST),
    id2label=ID2LABEL,
    label2id=LABEL2ID,
).to(DEVICE)

print(f'Model params: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# ── seqeval F1 metric ────────────────────────────────────────────────────────

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [ID2LABEL[pred] for (pred, label) in zip(prediction, label) if label != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [ID2LABEL[label] for (pred, label) in zip(prediction, label) if label != -100]
        for prediction, label in zip(predictions, labels)
    ]

    return {
        'precision': precision_score(true_labels, true_predictions),
        'recall':    recall_score(true_labels, true_predictions),
        'f1':        f1_score(true_labels, true_predictions),
    }

In [ ]:
# ── Training Arguments ───────────────────────────────────────────────────────
training_args = TrainingArguments(
    output_dir='../models/layoutlmv3_finetuned',
    evaluation_strategy='epoch',
    save_strategy='epoch',
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=20,
    learning_rate=1e-5,
    warmup_ratio=0.1,
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    logging_steps=50,
    save_total_limit=3,
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=processor,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

print('Trainer configured. Uncomment trainer.train() to start fine-tuning.')

In [ ]:
# ── TRAINING (uncomment to run) ───────────────────────────────────────────────
# trainer.train()

# ── EVALUATION ───────────────────────────────────────────────────────────────
# results = trainer.evaluate()
# print(f"Val F1: {results['eval_f1']:.4f}")
# print(f"Val Precision: {results['eval_precision']:.4f}")
# print(f"Val Recall: {results['eval_recall']:.4f}")
# print(f'Target: F1 >= 0.80')
print('Uncomment training/evaluation cells to run.')